In [10]:
# ==============================================================================
# CELL 1: CÀI ĐẶT
# ==============================================================================
!pip install yfinance -q
!pip install stockstats -q
!pip install pandas -q

import pandas as pd
import numpy as np
import yfinance as yf
from stockstats import StockDataFrame as Sdf

print("✅ Đã cài đặt xong.")

✅ Đã cài đặt xong.


In [11]:
# ==============================================================================
# CELL 2: TẢI DỮ LIỆU VN30 (FIX LỖI TRÙNG LẶP)
# ==============================================================================
# Danh sách VN30
TICKER_LIST = [
    "ACB", "BCM", "BID", "BVH", "CTG", "FPT", "GAS", "GVR", "HDB", "HPG",
    "MBB", "MSN", "MWG", "PLX", "POW", "SAB", "SHB", "SSB", "SSI", "STB",
    "TCB", "TPB", "VCB", "VHM", "VIB", "VIC", "VJC", "VNM", "VPB", "VRE"
]

# Thêm đuôi .VN
DOWNLOAD_LIST = [f"{x}.VN" for x in TICKER_LIST]

# Thời gian bạn chọn
START_DATE = '2018-01-01'
END_DATE = '2025-12-21'

print(f"📥 Đang tải dữ liệu {len(TICKER_LIST)} mã (2018 - 2025)...")

try:
    # Tải dữ liệu hàng loạt
    raw_data = yf.download(DOWNLOAD_LIST, start=START_DATE, end=END_DATE, group_by='ticker', ignore_tz=True)

    final_dfs = []

    for tic in TICKER_LIST:
        try:
            # Lấy dữ liệu từng mã
            df = raw_data[f"{tic}.VN"].copy()
        except KeyError:
            print(f"⚠️ Không tìm thấy dữ liệu cho {tic}")
            continue

        # Nếu ít dữ liệu quá thì bỏ qua
        if len(df) < 10:
            continue

        # --- XỬ LÝ RIÊNG TỪNG MÃ (QUAN TRỌNG) ---
        # 1. Reset index
        df = df.reset_index()

        # 2. Đổi tên cột
        df.columns = [c.lower() for c in df.columns]

        # 3. Xử lý giá đóng cửa (ưu tiên Adj Close)
        if 'adj close' in df.columns:
            df['close'] = df['adj close']
            # Xóa cột thừa để tránh nhầm lẫn sau này
            if 'adj close' in df.columns: del df['adj close']

        # 4. Thêm cột mã chứng khoán
        df['tic'] = tic

        # 5. Chọn cột chuẩn
        # Kiểm tra xem các cột có tồn tại không trước khi chọn
        required_cols = ['date', 'tic', 'open', 'high', 'low', 'close', 'volume']
        # Đôi khi yahoo trả về 'Date' viết hoa, đã lower() ở bước 2 nên là 'date'

        # Lọc cột
        df = df[[c for c in required_cols if c in df.columns]]

        # 6. Lấp đầy dữ liệu trống (FFILL/BFILL) - LÀM Ở ĐÂY ĐỂ KHÔNG BỊ TRỘN MÃ
        df = df.ffill().bfill()

        final_dfs.append(df)
        print(f"✅ {tic}: {len(df)} dòng")

    # Gộp tất cả lại
    full_df = pd.concat(final_dfs, ignore_index=True)
    full_df['date'] = pd.to_datetime(full_df['date'])
    full_df = full_df.sort_values(['date', 'tic']).reset_index(drop=True)

    # ==============================================================================
    # CELL 3: TÍNH CHỈ BÁO & LƯU FILE
    # ==============================================================================
    print("\n⚙️ Đang tính chỉ báo kỹ thuật...")
    INDICATORS = ['macd', 'rsi_30', 'cci_30', 'dx_30']

    processed_dfs = []
    for tic in full_df.tic.unique():
        tic_df = full_df[full_df.tic == tic].copy()
        stock = Sdf.retype(tic_df.copy())

        for ind in INDICATORS:
            try:
                tic_df[ind] = stock[ind]
                tic_df[ind] = tic_df[ind].replace([np.inf, -np.inf], np.nan).fillna(0)
            except:
                tic_df[ind] = 0
        processed_dfs.append(tic_df)

    final_train_df = pd.concat(processed_dfs, ignore_index=True)
    final_train_df = final_train_df.fillna(0)

    # Lưu file
    filename = "vietnam_vn30_data.csv"
    final_train_df.to_csv(filename, index=False)

    print(f"\n🎉 THÀNH CÔNG! Đã lưu file '{filename}'")
    print(f"📊 Tổng số dòng: {len(final_train_df)}")

    # In ra 5 dòng đầu để kiểm tra (ACB và BCM phải khác nhau!)
    print(final_train_df.head(5))

except Exception as e:
    print(f"❌ Có lỗi xảy ra: {e}")

📥 Đang tải dữ liệu 30 mã (2018 - 2025)...


YF.download() has changed argument auto_adjust default to True
[*********************100%***********************]  30 of 30 completed


✅ ACB: 1989 dòng
✅ BCM: 1989 dòng
✅ BID: 1989 dòng
✅ BVH: 1989 dòng
✅ CTG: 1989 dòng
✅ FPT: 1989 dòng
✅ GAS: 1989 dòng
✅ GVR: 1989 dòng
✅ HDB: 1989 dòng
✅ HPG: 1989 dòng
✅ MBB: 1989 dòng
✅ MSN: 1989 dòng
✅ MWG: 1989 dòng
✅ PLX: 1989 dòng
✅ POW: 1989 dòng
✅ SAB: 1989 dòng
✅ SHB: 1989 dòng
✅ SSB: 1989 dòng
✅ SSI: 1989 dòng
✅ STB: 1989 dòng
✅ TCB: 1989 dòng
✅ TPB: 1989 dòng
✅ VCB: 1989 dòng
✅ VHM: 1989 dòng
✅ VIB: 1989 dòng
✅ VIC: 1989 dòng
✅ VJC: 1989 dòng
✅ VNM: 1989 dòng
✅ VPB: 1989 dòng
✅ VRE: 1989 dòng

⚙️ Đang tính chỉ báo kỹ thuật...

🎉 THÀNH CÔNG! Đã lưu file 'vietnam_vn30_data.csv'
📊 Tổng số dòng: 59670
        date  tic         open         high          low        close  \
0 2018-01-02  ACB  6642.337792  6983.430580  6606.433218  6965.478516   
1 2018-01-03  ACB  6965.478878  7019.335520  6821.860573  6947.526367   
2 2018-01-04  ACB  6947.526006  6983.430580  6893.669367  6965.478516   
3 2018-01-05  ACB  7001.383090  7091.144303  6893.669367  6965.478516   
4 2018-01-08  ACB 